# Sesión 3 · Autenticación y autorización

**Curso MCP · servidores remotos** — notebook 3 de 4

Tu servidor lleva dos sesiones abierto a internet con `--allow-unauthenticated`. Cualquiera
con la URL puede lanzar consultas **que pagas tú**. Hoy lo cerramos.

Esta es la sesión que separa un juguete de un conector que puede tocar datos reales.

| Bloque | Minutos |
|---|:--:|
| Por qué authless deja de valer | 15 |
| OAuth en MCP: servidor y cliente | 40 |
| El puente a BigQuery | 35 |

> ### ⚠️ Esta sesión necesita preparación previa
>
> Es la única del curso cuyas celdas **no se pueden ejecutar improvisando**. OAuth requiere
> cosas que se piden con antelación y que a veces tardan en aprobarse.
>
> **Antes de la clase hay que tener:**
>
> - [ ] Un **OAuth 2.0 Client ID** creado en *APIs y servicios → Credenciales* de la consola de
>       GCP, con la pantalla de consentimiento configurada.
> - [ ] La cuenta de servicio de Cloud Run con **`roles/iam.serviceAccountTokenCreator`** sobre
>       las cuentas que vaya a suplantar, si se hace la parte de suplantación de identidad.
> - [ ] Un **documento CIMD** publicado en una URL bajo tu control, si vas a probar el flujo de
>       cliente completo.
>
> **Los fragmentos de este notebook están escritos contra la API real del SDK 2.0.0, pero el
> flujo de extremo a extremo no se ha verificado en este repositorio** porque depende de esas
> credenciales. Trátalos como el esqueleto correcto, no como algo que funcionará a la primera:
> **pruébalos tú antes de dar la sesión**. Los nombres de proyecto, cuentas y dominios que
> aparecen son marcadores.
>
> Lo que sí se puede recorrer sin nada configurado: los bloques 1 y 2 hasta la teoría, y la
> comprobación del `401` con `WWW-Authenticate` al final.

## 1. Lo que expone un servidor abierto

Antes de arreglarlo, mide el problema. Tu servidor ahora mismo permite a cualquiera:

- **Gastar tu dinero.** Cada consulta escanea bytes que se te facturan. El límite del módulo
  anterior protege del error accidental, no del abuso deliberado.
- **Leer todo lo que el servidor pueda leer.** No hay «usuario»: hay una cuenta de servicio
  con sus permisos, y quien llega a la URL hereda todos.
- **Agotarte el servicio.** Cloud Run escala, y escalar cuesta.

El tercero suele sorprender: **incluso con datos públicos**, un servidor MCP abierto es una
factura abierta.

In [ ]:
# Compruébalo: sin credencial de ningún tipo, respondes a todo el mundo.
!pip install --quiet "mcp==2.0.0" httpx

MI_URL = "https://curso-mcp-XXXXX.europe-west1.run.app/mcp"  # ← EDITAR

import httpx
r = httpx.post(MI_URL, json={
    "jsonrpc":"2.0","id":1,"method":"tools/list",
    "params":{"_meta":{"io.modelcontextprotocol/protocolVersion":"2026-07-28"}},
}, headers={"Content-Type":"application/json","Accept":"application/json, text/event-stream",
            "Mcp-Method":"tools/list","Mcp-Name":"curso"}, timeout=30)
print(r.status_code, "· sin un solo token")

## 2. OAuth en MCP

MCP no inventa un sistema de autenticación: usa **OAuth 2.1**, con un reparto de papeles que
conviene tener claro porque es donde se lía todo el mundo.

| Papel | Quién lo hace | Qué le toca |
|---|---|---|
| **Resource Server** | **Tu servidor MCP** | Validar el token que llega y decidir qué deja hacer |
| **Authorization Server** | Google, Auth0, Entra… | Autenticar a la persona y emitir tokens |
| **Cliente** | El host (Claude, tu notebook) | Conseguir el token y mandarlo |

**Tu servidor no autentica a nadie.** Recibe un token, comprueba que es válido y actúa en
consecuencia. Montar tu propio servidor de autorización es casi siempre el camino equivocado.

### Cómo se encuentran las piezas

El cliente no sabe de antemano quién emite los tokens de tu servidor. El protocolo lo resuelve
así:

1. El cliente llama sin token. Tu servidor responde **`401`** con una cabecera
   `WWW-Authenticate` que apunta a los metadatos del recurso protegido.
2. El cliente lee esos metadatos y descubre **qué servidor de autorización** usas.
3. Consigue un token contra ese servidor.
4. Repite la llamada con `Authorization: Bearer …`.

Todo eso lo hace el SDK del cliente solo. Lo que tú declaras es el paso 1.

### Registro del cliente: CIMD

Para pedir un token, el cliente necesita estar registrado en el servidor de autorización. En
MCP la vía preferente son los **Client ID Metadata Documents (CIMD)**: el cliente publica un
documento con sus metadatos en una URL bajo su control, y **esa URL es su identificador**.
Sin registro previo, sin secretos que repartir.

> La alternativa histórica es el registro dinámico (DCR), donde cada cliente se da de alta al
> conectarse. Sigue funcionando por compatibilidad, pero para un conector nuevo la
> recomendación es CIMD.

### El lado servidor

Declarar que estás protegido son dos objetos:

```python
from mcp.server.auth.settings import AuthSettings
from mcp.server.auth.provider import AccessToken, TokenVerifier

class VerificadorGoogle(TokenVerifier):
    async def verify_token(self, token: str) -> AccessToken | None:
        datos = await validar_id_token_de_google(token)   # tuyo
        if datos is None:
            return None
        return AccessToken(
            token=token,
            client_id=datos["aud"],
            scopes=["bigquery.read"],
            subject=datos["email"],     # ← la identidad que usaremos en el bloque 3
            expires_at=datos["exp"],
        )

mcp = MCPServer(
    "curso-mcp-bigquery",
    token_verifier=VerificadorGoogle(),
    auth=AuthSettings(
        issuer_url="https://accounts.google.com",
        resource_server_url="https://TU-SERVICIO.run.app",
        required_scopes=["bigquery.read"],
    ),
)
```

Lo que devuelve `verify_token` no es un sí o un no: es **quién es** quien llama. El campo
`subject` es la pieza que hace posible todo el bloque siguiente.

In [ ]:
# Verificador real de ID tokens de Google, para que veas que no hay magia.
!pip install --quiet google-auth

from google.oauth2 import id_token
from google.auth.transport import requests as gauth_requests
from mcp.server.auth.provider import AccessToken, TokenVerifier

CLIENT_ID = "TU-OAUTH-CLIENT-ID.apps.googleusercontent.com"  # ← EDITAR

class VerificadorGoogle(TokenVerifier):
    async def verify_token(self, token: str) -> AccessToken | None:
        try:
            datos = id_token.verify_oauth2_token(token, gauth_requests.Request(), CLIENT_ID)
        except ValueError:
            return None   # firma inválida, expirado o audiencia equivocada
        return AccessToken(
            token=token,
            client_id=datos["aud"],
            scopes=["bigquery.read"],
            subject=datos.get("email"),
            expires_at=datos.get("exp"),
        )

print("Verificador listo. `subject` llevará el email de quien llama.")

## 3. El puente a BigQuery

Aquí está el contenido que no encontrarás escrito en ningún otro sitio, y el motivo de que
esta sesión ocupe dos horas.

Ya sabes **quién** llama. Falta que eso signifique algo cuando el servidor toque BigQuery.

### El problema de la llave maestra

Tu servidor corre en Cloud Run con una cuenta de servicio. Esa cuenta tiene permisos sobre
BigQuery. Si el código hace `bigquery.Client()` sin más, **todo el mundo consulta con los
permisos de la cuenta de servicio**, autenticado o no.

Has puesto una puerta con cerradura y detrás has dejado la llave maestra colgada.

### Tres formas de resolverlo

| Estrategia | Cómo | Cuándo |
|---|---|---|
| **Filtrado en el servidor** | La cuenta de servicio lee todo; el servidor recorta según `subject` | Reglas simples, pocos perfiles |
| **Suplantación de identidad** | El servidor pide credenciales *en nombre de* quien llama y BigQuery aplica sus permisos | Cuando GCP ya es la fuente de verdad |
| **Cuenta por perfil** | Varias cuentas de servicio con permisos distintos; se elige según el token | Perfiles gruesos y estables |

La segunda es la buena cuando tus usuarios ya viven en Google Cloud: **quien decide es IAM,
no tu código**. Y eso importa porque el permiso deja de depender de que no te hayas dejado un
`if`.

In [ ]:
# Suplantación: el servidor actúa EN NOMBRE de quien llama.
# Requiere que la cuenta de servicio de Cloud Run tenga
# `roles/iam.serviceAccountTokenCreator` sobre la cuenta destino.
from google.auth import impersonated_credentials, default
from google.cloud import bigquery

def cliente_para(subject: str) -> bigquery.Client:
    origen, _ = default()
    destino = f"mcp-{subject.split('@')[0]}@TU-PROYECTO.iam.gserviceaccount.com"
    credenciales = impersonated_credentials.Credentials(
        source_credentials=origen,
        target_principal=destino,
        target_scopes=["https://www.googleapis.com/auth/bigquery"],
        lifetime=600,
    )
    return bigquery.Client(credentials=credenciales)

print("Ahora BigQuery aplica los permisos del usuario, no los del servidor.")

### Y en el tool

El contexto de la petición lleva la identidad ya verificada, así que el tool no vuelve a
comprobar nada: **usa** la identidad.

```python
from mcp.server.auth.middleware.auth_context import get_access_token

@mcp.tool(description="Consulta el dataset con TUS permisos.")
async def consultar(sql: str, ctx: Context) -> list[dict]:
    token = get_access_token()
    if token is None or token.subject is None:
        raise RuntimeError("Esta operación requiere identificarse.")
    cliente = cliente_para(token.subject)      # ← permisos de quien llama
    return [dict(f) for f in cliente.query(sql).result(max_results=50)]
```

**El tool ya no decide quién puede ver qué.** Lo decide IAM. Si mañana revocan el acceso de
alguien a un dataset, tu servidor se entera sin desplegar nada.

In [ ]:
# Redespliegue cerrando la puerta.
PROYECTO, REGION, SERVICIO = "tu-proyecto", "europe-west1", "curso-mcp"

!gcloud run deploy {SERVICIO} \
  --source . \
  --region {REGION} \
  --no-allow-unauthenticated \
  --set-env-vars CURSO_MCP_AUTH=1,CURSO_MCP_CLIENT_ID={CLIENT_ID} \
  --quiet

In [ ]:
# Y comprobación de que ahora sí protege.
import httpx
r = httpx.post(MI_URL, json={
    "jsonrpc":"2.0","id":1,"method":"tools/list",
    "params":{"_meta":{"io.modelcontextprotocol/protocolVersion":"2026-07-28"}},
}, headers={"Content-Type":"application/json","Accept":"application/json, text/event-stream",
            "Mcp-Method":"tools/list","Mcp-Name":"curso"}, timeout=30)

print(r.status_code)
print("WWW-Authenticate:", r.headers.get("www-authenticate"))

Ese `401` con su cabecera `WWW-Authenticate` **no es un fallo**: es el primer paso del baile.
Es como tu servidor le dice a un cliente que no conoce dónde tiene que ir a por un token.

In [ ]:
# El cliente completo, ya autenticado. El SDK hace el descubrimiento y el flujo solo.
from mcp import Client
from mcp.client.auth import OAuthClientProvider

async def con_identidad():
    proveedor = OAuthClientProvider(
        server_url=MI_URL,
        client_metadata_url="https://tu-dominio.example/mcp-client.json",  # CIMD
    )
    async with Client(MI_URL, auth=proveedor) as c:
        lista = await c.list_tools()
        print("Autenticado. Tools visibles:", [t.name for t in lista.tools])

# await con_identidad()   # ← descomenta cuando tengas tu CIMD publicado

## Ejercicios

1. **Perfiles.** Haz que `subject` con dominio `@alumnos.…` solo pueda usar `listar_tablas` y
   `describir_tabla`, y no `consultar`. Hazlo dos veces: filtrando en el servidor y con IAM.
   Compara qué pasa cuando alguien cambia de rol.
2. **El 401 bien hecho.** Comprueba que la cabecera `WWW-Authenticate` apunta a tus metadatos
   y que un cliente ajeno puede seguir el rastro sin documentación previa.
3. **Token caducado.** Manda uno expirado a propósito y verifica que el error distingue entre
   «no te conozco» y «no te dejo».

## En la próxima sesión

**Extensiones:** cómo el protocolo crece sin romperse, las de autorización, MCP Apps con un
widget sobre tus datos, y construir la tuya propia.